# Week 5 Homework: Multi-Bit Comparison, Calibration Recovery, Deployment Memo

**ECBS5200 — Practical Deep Learning Engineering**

The lab produced six measurements and a per-tier picture at int4. The
homework extends that: per-tier Δacc at int8 too (so you can compare the
int8 and int4 damage patterns), temperature scaling fit on one half of
val and evaluated on the other (does it generalize, or does it overfit),
and named-class trajectories across all three precisions.

Then the memo. Five sections, 100 points. Rubric at
`assessments/week5_memo_rubric.md`. The Excellent-band descriptors
describe what mastery of each prompt looks like — use them as targets,
not gatekeepers.

**Expected time:** ~5-6 hours including the memo. The heavy compute is
done — lab saved the logits. The homework is analysis and writing.

**Point breakdown (memo):** 20 / 25 / 15 / 30 / 10 across five sections.
Section 4 (operational decision) is weighted heaviest because it's where
measurement discipline becomes engineering judgment.

**How to use this notebook:**
- **GUIDED** cells run as-is.
- **INTERACTIVE** cells require you to fill in values or write answers.
- `___` is a NameError-producing placeholder.
- The memo sections are at the end. Fill them in as you go.

## Kaggle setup

1. **Persistence** → "Variables and Files"
2. **Internet** → On
3. **Accelerator** → None (this homework reads the saved lab logits and
   runs entirely on CPU — no GPU needed)
4. **Attach your lab notebook's outputs** — in the right sidebar click
   **Add Input** → **Your Work** → pick the version of your `week5_lab`
   notebook you ran. Kaggle mounts its four output files under
   `/kaggle/input/<notebook-slug>/` and the discovery code below finds
   them regardless of the slug. (No need to create a separate Kaggle
   dataset.)

In [ ]:
import subprocess, sys, os

if os.path.exists('/kaggle/working'):
    os.environ['HF_HOME'] = '/kaggle/working/.hf_cache'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "bitsandbytes>=0.43", "transformers>=4.53", "datasets", "accelerate",
    "scikit-learn", "matplotlib", "pandas", "tqdm", "peft",
])
print("Packages installed.")

from huggingface_hub import login
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token)

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

REPO_PATH = Path("../..").resolve()
if not (REPO_PATH / "utils" / "data_utils.py").exists():
    REPO_PATH = Path("/tmp/course")
    if not REPO_PATH.exists():
        subprocess.check_call(["git", "clone", "--depth", "1",
            "https://github.com/earino/applied-deep-learning.git", str(REPO_PATH)])
sys.path.insert(0, str(REPO_PATH))

from utils.data_utils import load_course_data, LABEL_LIST, NUM_LABELS, MAX_LENGTH

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

---
## Part 1: Load lab artifacts (~5 min)

The lab saved four files; this homework reads them and analyzes. If you
can't find them, see the recovery path in the comment below.
---

In [ ]:
# GUIDED: Locate lab predictions.
# The lab saved four files. Attach them to this notebook as a Kaggle dataset
# (any slug — we glob for the filename) or put them in the working directory.

def find_lab_file(name):
    """Look in cwd first, then scan /kaggle/input/* for the filename."""
    local = Path(name)
    if local.exists():
        return local
    for p in Path("/kaggle/input").glob(f"*/{name}"):
        return p
    for p in Path("/kaggle/input").rglob(name):
        return p
    return None

lab_npz = find_lab_file("week5_lab_predictions.npz")

if lab_npz is None:
    # The homework's job is analysis, not re-measurement, so we don't
    # duplicate the lab's quantized inference loop here. If you've lost
    # your lab outputs, the recovery path is:
    #
    #   1. Re-open YOUR completed copy of the week5 lab notebook on Kaggle
    #      (the one with your filled-in answers — running a fresh student
    #      copy will NameError on the `___` placeholders).
    #   2. Run all cells. The final cell saves four files:
    #          week5_lab_predictions.npz   <- the one this notebook needs
    #          week5_lab_summary.csv
    #          week5_lab_ece.csv
    #          week5_lab_tier_deltas.json
    #   3. Either (a) keep this homework attached to that lab via Kaggle's
    #      "Add Input → Notebook Output", or (b) download the four files
    #      and upload them as a Kaggle dataset, then attach.
    #   4. Re-run this cell. find_lab_file() will pick them up automatically.
    #
    # If you've genuinely lost the ability to re-run the lab (checkpoints
    # gone, etc.), email the instructor before the deadline.
    raise FileNotFoundError(
        "week5_lab_predictions.npz not found in cwd or under /kaggle/input. "
        "See the comment directly above this line for the recovery path."
    )

arr = np.load(lab_npz)
val_labels = arr["val_labels"]
val_tiers = arr["val_tiers"]
logits = {
    "encoder": {
        "fp16": arr["encoder_fp16_logits"],
        "int8": arr["encoder_int8_logits"],
        "int4": arr["encoder_int4_logits"],
    },
    "decoder": {
        "fp16": arr["decoder_fp16_logits"],
        "int8": arr["decoder_int8_logits"],
        "int4": arr["decoder_int4_logits"],
    },
}
print(f"Loaded {lab_npz}")
print(f"  val n = {len(val_labels)}, tiers: {dict(Counter(val_tiers.tolist()))}")

# Derive probs + preds from logits — homework analysis doesn't need logits back on GPU
probs, preds = {}, {}
for m in logits:
    probs[m], preds[m] = {}, {}
    for p in logits[m]:
        pr = torch.softmax(torch.from_numpy(logits[m][p]), dim=-1).numpy()
        probs[m][p] = pr
        preds[m][p] = pr.argmax(-1)

In [ ]:
# GUIDED: Load the companion CSVs + JSON using the same discovery pattern
summary_path = find_lab_file("week5_lab_summary.csv")
ece_path = find_lab_file("week5_lab_ece.csv")
tier_deltas_path = find_lab_file("week5_lab_tier_deltas.json")

summary_df = pd.read_csv(summary_path)
ece_df = pd.read_csv(ece_path)
with open(tier_deltas_path) as f:
    tier_deltas_lab = json.load(f)

print("Lab summary:")
print(summary_df.to_string(index=False))
print("\nLab ECE table:")
print(ece_df.to_string(index=False))

---
## Part 2: Per-tier Δacc — int8 AND int4 (~30 min, analysis only)

The lab computed bootstrap tier Δacc for int4 only. Extend to int8 so you
can compare the tier patterns across precisions — a key question for
Memo Section 2.
---

In [ ]:
# GUIDED: Bootstrap helper — same as lab
def bootstrap_tier_deltas(preds_ref, preds_test, labels, tiers,
                           n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    out = {}
    for tier in ["head", "mid", "tail"]:
        idxs = np.where(tiers == tier)[0]
        n = len(idxs)
        deltas = []
        for _ in range(n_boot):
            bi = rng.choice(idxs, n, replace=True)
            a_ref = (preds_ref[bi] == labels[bi]).mean()
            a_test = (preds_test[bi] == labels[bi]).mean()
            deltas.append(a_test - a_ref)
        deltas = np.array(deltas)
        out[tier] = {
            "n": int(n), "mean": float(deltas.mean()),
            "ci_low": float(np.percentile(deltas, 2.5)),
            "ci_high": float(np.percentile(deltas, 97.5)),
        }
    return out

In [ ]:
# INTERACTIVE: Compute tier Δacc at int8 AND int4 for both models
# Fill both precisions. Keep the int4 from the lab for a sanity check, but
# recompute so the homework table is internally consistent (same seed, same code).

tier_deltas = {"encoder": {}, "decoder": {}}
for model_name in ["encoder", "decoder"]:
    preds_fp16 = preds[model_name]["fp16"]
    for precision in ["int8", "int4"]:
        preds_test = preds[model_name][precision]
        tier_deltas[model_name][precision] = ___  # bootstrap_tier_deltas(...)

In [ ]:
# GUIDED: Display — four blocks total (2 models × 2 precisions)
print(f"{'model':<10}{'prec':<6}{'tier':<8}{'n':>6}{'Δacc':>10}{'95% CI':>22}")
print("-" * 62)
for model_name in ["encoder", "decoder"]:
    for precision in ["int8", "int4"]:
        td = tier_deltas[model_name][precision]
        for tier in ["head", "mid", "tail"]:
            t = td[tier]
            ci = f"[{t['ci_low']:+.4f}, {t['ci_high']:+.4f}]"
            excl = "  *" if (t["ci_low"] > 0 or t["ci_high"] < 0) else ""
            print(f"{model_name:<10}{precision:<6}{tier:<8}{t['n']:>6}"
                  f"{t['mean']:+10.4f}{ci:>22}{excl}")
        print()

In [ ]:
# GUIDED: Bar-chart visualization — tiers on x, one subplot per model, bars for int8 vs int4
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
tier_order = ["head", "mid", "tail"]
x = np.arange(len(tier_order))
w = 0.35
for ax, model_name in zip(axes, ["encoder", "decoder"]):
    for i, precision in enumerate(["int8", "int4"]):
        td = tier_deltas[model_name][precision]
        means = [td[t]["mean"] for t in tier_order]
        errs_lo = [td[t]["mean"] - td[t]["ci_low"] for t in tier_order]
        errs_hi = [td[t]["ci_high"] - td[t]["mean"] for t in tier_order]
        ax.bar(x + (i - 0.5) * w, means, w, label=precision,
               yerr=[errs_lo, errs_hi], capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(tier_order)
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set_title(model_name)
    ax.set_ylabel("Δacc vs fp16 (95% CI)")
    ax.legend()
plt.tight_layout()
plt.show()

### INTERACTIVE: Compare int8 and int4 tier patterns

Look at the four tier-blocks.

1. For each model, describe the relationship between its int8 tier
   pattern and its int4 tier pattern. (Proportionally larger at int4?
   Different shape? Same tier moving in different directions?)
2. Are there (model, tier) rows where int8 and int4 disagree on
   whether the CI excludes zero? What would that mean for a
   precision-choice decision on that tier?
3. Across all twelve (2 models × 2 precisions × 3 tiers) rows, list
   which CIs cleanly exclude zero and which clearly cross zero.

YOUR ANSWERS:

1.

2.

3. CI excludes zero:
   CI crosses zero:


---
## Part 3: Temperature scaling — fit-then-evaluate on split val (~30 min)

The lab fit T on all of val and evaluated ECE on the same data. That's a
reasonable sanity check but it's not how you'd deploy: you'd fit T on
a held-out calibration set and apply it to new data.

Split the val set in half deterministically. Fit T on half A. Evaluate
ECE on half B *after applying* that T. Do for all six configs. If T from
half A generalizes, ECE on half B should drop by a similar amount to
what we saw in the lab. If it overfits, B's ECE won't improve as much
(or might get worse).
---

In [ ]:
# GUIDED: Deterministic split — every-other-example so tier distribution is preserved
n_val = len(val_labels)
idx_A = np.arange(0, n_val, 2)       # indices 0, 2, 4, ...
idx_B = np.arange(1, n_val, 2)       # indices 1, 3, 5, ...
print(f"Split A: {len(idx_A)}   Split B: {len(idx_B)}")

In [ ]:
# GUIDED: ECE + temperature-scaling helpers (same as lab / Week 4)
def compute_ece(probs, labels, n_bins=15):
    confidences = probs.max(axis=-1)
    preds_local = probs.argmax(axis=-1)
    accuracies = (preds_local == labels).astype(float)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece, n = 0.0, len(labels)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() > 0:
            bin_acc = accuracies[mask].mean()
            bin_conf = confidences[mask].mean()
            ece += (mask.sum() / n) * abs(bin_acc - bin_conf)
    return float(ece)

def temperature_scale(logits, labels, max_iter=50, tol=1e-4):
    """Fit T > 0 minimizing NLL via LBFGS in log space (T = exp(log_T))."""
    logits_t = torch.from_numpy(logits).float()
    labels_t = torch.from_numpy(labels).long()
    log_T = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.01, max_iter=max_iter)
    def closure():
        opt.zero_grad()
        T = torch.exp(log_T)
        loss = F.cross_entropy(logits_t / T, labels_t)
        loss.backward()
        return loss
    opt.step(closure)
    final_grad = abs(log_T.grad.item()) if log_T.grad is not None else float("inf")
    if final_grad > tol:
        print(f"    [warn] temperature_scale: |grad|={final_grad:.2e} > {tol:.0e} after {max_iter} LBFGS steps; T may be sub-optimal")
    return float(torch.exp(log_T.detach()).item())

In [ ]:
# INTERACTIVE: For each (model, precision), fit T on half A, evaluate on half B.
# Fill the inner logic — what you're scaling, what you're measuring.

split_rows = []
for model_name in ["encoder", "decoder"]:
    for precision in ["fp16", "int8", "int4"]:
        L = logits[model_name][precision]
        # Half A — fit T
        T = temperature_scale(L[idx_A], val_labels[idx_A])
        # Half B — ECE with and without the fitted T
        probs_B_pre = torch.softmax(torch.from_numpy(L[idx_B]), dim=-1).numpy()
        # Apply fitted T to half B logits, then softmax for probs
        probs_B_post = ___
        ece_B_pre = compute_ece(probs_B_pre, val_labels[idx_B])
        ece_B_post = compute_ece(probs_B_post, val_labels[idx_B])
        split_rows.append({
            "model": model_name, "precision": precision,
            "T_fit_on_A": round(T, 3),
            "ece_B_pre":  round(ece_B_pre,  4),
            "ece_B_post": round(ece_B_post, 4),
            "delta": round(ece_B_post - ece_B_pre, 4),
        })

split_ece_df = pd.DataFrame(split_rows)
print(split_ece_df.to_string(index=False))

### INTERACTIVE: Did temperature scaling generalize?

Compare the `split_ece_df` (fit on A, evaluate on B) to the `ece_df`
from the lab (fit on full val, evaluate on full val).

1. For each (model, precision), compare the ECE drop on half B against
   the drop you measured in the lab. Roughly matching, smaller, or
   larger? What would each of those outcomes imply about T as a fitted
   parameter?
2. Considering the split-val result, would you give a different answer
   to the lab question "can T-scaling recover calibration under int4"
   than you gave yesterday? If yes, name the (model, precision) and
   the new answer. If no, explain what the split-val result added (or
   didn't).

YOUR ANSWERS:

1.

2.


---
## Part 4: Named-class trajectories (~30 min)

The lab tracked three classes by name: *"Incorrect information on your
report"* (head), *"Managing, opening, or closing your mobile wallet
account"* (mid), and *"Cash advance fee"* (tail).

Now look at each one's dynamics as precision drops. For each named class
and each model, compute:
- accuracy on that class (fraction of val examples with true label = cls
  correctly predicted)
- mean P(true class) — the model's average probability placed on the
  correct answer across all val examples of that class

For head and mid these are reliable numbers. For tail with n_val = 2,
accuracy is 0.0, 0.5, or 1.0 — treat tail dynamics here as directional
only. The point of the tail row isn't to estimate accuracy, it's to
watch what the model's confidence on a barely-predictable class does
as you compress.
---

In [ ]:
# GUIDED: Named classes (same as lab; resolve via substring if needed)
NAMED_CLASSES = {
    "head": "Incorrect information on your report",
    "mid":  "Managing, opening, or closing your mobile wallet account",
    "tail": "Cash advance fee",
}

def name_to_id(name):
    if name in LABEL_LIST:
        return LABEL_LIST.index(name)
    hits = [i for i, n in enumerate(LABEL_LIST) if name.lower() in n.lower()]
    if not hits:
        hits = [i for i, n in enumerate(LABEL_LIST)
                if name.split(",")[0].lower() in n.lower()]
    return hits[0]

NAMED_IDS = {tier: name_to_id(n) for tier, n in NAMED_CLASSES.items()}
print("Named classes:", NAMED_IDS)

In [ ]:
# INTERACTIVE: Build the trajectory table
# For each (tier, model, precision), compute accuracy and mean P(true)
# on val examples whose true label == NAMED_IDS[tier].

trajectory_rows = []
for tier, cls_id in NAMED_IDS.items():
    mask = val_labels == cls_id
    n_cls = int(mask.sum())
    for model_name in ["encoder", "decoder"]:
        for precision in ["fp16", "int8", "int4"]:
            pr = probs[model_name][precision]
            pd_preds = preds[model_name][precision]
            # accuracy = fraction of examples whose prediction equals cls_id
            acc = ___  # (pd_preds[mask] == cls_id).mean()
            # mean P(true class) = mean of pr[mask, cls_id]
            p_true_mean = ___
            trajectory_rows.append({
                "tier": tier, "class": LABEL_LIST[cls_id], "class_id": cls_id,
                "n_val": n_cls, "model": model_name, "precision": precision,
                "accuracy": round(float(acc), 4),
                "mean_p_true": round(float(p_true_mean), 4),
            })

trajectory_df = pd.DataFrame(trajectory_rows)
print(trajectory_df.to_string(index=False))

In [ ]:
# GUIDED: Plot the trajectory — mean P(true) across precisions, one line per (tier, model)
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
precisions_order = ["fp16", "int8", "int4"]
x = np.arange(len(precisions_order))
colors = {"encoder": "#2166AC", "decoder": "#B2182B"}
for ax, tier in zip(axes, ["head", "mid", "tail"]):
    for model_name in ["encoder", "decoder"]:
        sub = trajectory_df[(trajectory_df["tier"] == tier) &
                            (trajectory_df["model"] == model_name)]
        sub = sub.sort_values("precision",
                              key=lambda s: s.map({p: i for i, p in enumerate(precisions_order)}))
        ax.plot(x, sub["mean_p_true"].values, marker="o",
                color=colors[model_name], linewidth=2, label=model_name)
    ax.set_xticks(x)
    ax.set_xticklabels(precisions_order)
    ax.set_ylim(0, 1)
    ax.set_title(f"{tier.upper()}: {NAMED_CLASSES[tier][:42]}")
    ax.set_ylabel("mean P(true class)")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

### INTERACTIVE: Read the trajectories

Look at the three plots.

1. Describe the HEAD-class trajectory for each model across fp16 → int8
   → int4. Does the named-class trajectory look like the head *tier*
   average from Part 2, or does this specific class behave differently?
2. Describe the MID-class trajectory for each model. Any precision
   where the two models diverge?
3. On the TAIL class with n_val = 2, accuracy is constrained to 0 /
   0.5 / 1.0 — report those values but read mean-P(true) as the more
   informative signal. What does confidence on this barely-predictable
   class do as you compress?

YOUR ANSWERS:

1. Head:
2. Mid:
3. Tail:


---
## Part 5: Deployment recommendation (~30 min)

You now have:
- Six-point summary table (lab)
- Tier Δacc at int8 and int4 (this homework)
- Split-val T-scaling generalization (this homework)
- Named-class trajectories (this homework)

Synthesize into a deployment recommendation for a specific constraint
envelope. This is the evidence backbone for Memo Section 4.
---

In [ ]:
# INTERACTIVE: Check each constraint against each of the six configs.
# The stated constraints (hypothetical):
#   - Hardware: single T4 GPU
#   - Throughput: 100 requests/second sustained, batched
#   - Quality floor: macro F1 >= 0.20
#   - Calibration ceiling: post-scaling ECE <= 0.05
#
# Throughput constraint translation: at batch size 32, you need median
# latency <= 1000 ms / 100 req * 32 batch = 320 ms/batch = 10 ms/example.
# (This is a simplification — real serving has request queueing too —
# but it's the right order of magnitude for the memo-level discussion.)

LATENCY_CEILING_MS_PER_EX = 10.0
F1_FLOOR = 0.20
ECE_CEILING_POST = 0.05

# For each row of summary_df, check each constraint. Write True/False.
# Join with ece_df to get ece_post.
check_df = summary_df.merge(
    ece_df[["model", "precision", "ece_post"]],
    on=["model", "precision"],
)
check_df["meets_latency"]  = ___  # latency_ms <= LATENCY_CEILING_MS_PER_EX
check_df["meets_f1"]       = ___  # macro_f1 >= F1_FLOOR
check_df["meets_ece"]      = ___  # ece_post <= ECE_CEILING_POST
check_df["meets_all"]      = (check_df["meets_latency"] &
                              check_df["meets_f1"] &
                              check_df["meets_ece"])
print(check_df.to_string(index=False))

### INTERACTIVE: Which constraint is binding?

For any config, a "binding" constraint is one it's closest to failing
(or has already failed). If a config comfortably meets all four, it
has no binding constraint; if it fails some, the failure gives you a
concrete relaxation candidate.

1. How many of the six configs meet ALL four constraints?
2. For any config that fails one or more constraints, list which
   constraints it fails and what would have to change to unblock it
   (different hardware, larger latency budget, different quantization
   tool, better calibration fit).
3. If you had to ship exactly one of the six on the stated constraints,
   which would you pick and what's the first thing you'd change if the
   constraint envelope could be renegotiated?

YOUR ANSWERS:

1.

2.

3.


---
## Part 6: Memo (100 points)

Five sections. Rubric at `assessments/week5_memo_rubric.md` — read the
Excellent-band descriptors before you write. Cite specific numbers from
your own runs.

General rubric expectations:
- **Evidence + reasoning**, not just evidence.
- **Intellectual honesty** — "I don't know, here's why, here's what
  I'd need" scores higher than overclaiming.
- **Hardware-awareness** — if your int8 numbers are T4-specific, say
  so. If you'd expect different results on AWQ + vLLM, say why.
- **Concise prose** — 2-3 pages max (not counting tables/figures).
---

### MEMO SECTION 1: Accuracy-Efficiency Frontier (20 points)

> Read the Pareto frontier you measured across fp16/int8/int4 for both
> models (quality vs latency AND quality vs VRAM). Identify the
> dominant and dominated points. What (model, precision) combination
> would you argue for in a deployment memo, given a specified workload,
> and why? Acknowledge hardware dependence — name one claim whose truth
> value would change if the hardware did.

**Evidence expected (from your own runs):**
- 6-row summary table or equivalent prose
- Explicit identification of Pareto structure you observed (dominant /
  dominated points if any, or an argument for why no point dominates)
- One claim about hardware-dependence (e.g., T4 int8 slowdown, AWQ on
  vLLM would look different)

**YOUR RESPONSE:**





### MEMO SECTION 2: Where the Damage Lands — Per-Tier Analysis (25 points)

> For both encoder and decoder, at both int8 and int4, report per-tier
> Δacc with the bootstrap CIs. What pattern do you observe? What does
> the macro F1 number alone hide? Distinguish real effects (CI excludes
> zero) from noise-limited findings (CI crosses zero) — treat each with
> appropriate modesty.

**Evidence expected:**
- Full tier-breakdown table (4 blocks: 2 models × 2 precisions)
- At least one claim about at least one named class (head / mid /
  tail) anchored in your trajectory plot
- Clear handling of CI width — don't overclaim narrow CIs as "proof,"
  don't dismiss wide CIs as "nothing"

**YOUR RESPONSE:**





### MEMO SECTION 3: Calibration Under Compression (15 points)

> Did ECE shift with quantization? Did temperature scaling recover
> calibration under int8 and int4? Did the fit generalize from half A
> to half B, or did it overfit? What does this mean for a deployment
> that uses confidence as a decision gate (e.g., route to human review
> below 0.8 confidence)?

**Evidence expected:**
- ECE at fp16/int8/int4 pre- AND post-scaling on your lab measurements
- Split-val comparison (this homework, Part 3)
- One sentence connecting to Week 4's calibration work
- A concrete deployment implication grounded in a specific number from
  your tables

**YOUR RESPONSE:**





### MEMO SECTION 4: Operational Decision Under Constraints (30 points)

> **Constraints:** single T4 GPU, 100 requests/second sustained, macro
> F1 floor 0.20, post-scaling ECE ceiling 0.05.
>
> Make a specific (model, precision) recommendation against these
> constraints. If no config meets all four, name which constraint
> you'd relax and justify why. Your argument should integrate at least
> three evidence sources (latency, per-tier F1, calibration, VRAM
> headroom). Cite at least one reading from `readings/week5/` where it
> bears on your decision.

**Evidence expected:**
- Constraint-by-constraint check (`check_df` from Part 5)
- Explicit identification of which constraint is binding for your
  chosen config, and which constraint you would relax if you could
- Acknowledgment of how the tool (bitsandbytes) and hardware (T4)
  shape your numbers — what a different tool/hardware stack would
  plausibly change
- Citation of at least one spotlight paper (AWQ, Lee 2025, LLM.int8,
  SmoothQuant) — by author, in-line

**YOUR RESPONSE:**





### MEMO SECTION 5: Revisiting Week 3 (10 points)

> Your Week 3 memo recommended an encoder or decoder. Does the Week 5
> quantization evidence reinforce that recommendation, complicate it,
> or shift it? What single finding most moved your view (or solidified
> it)? What evidence would further change your mind?

**Evidence expected:**
- A specific reference to your Week 3 position
- Identification of at least one Week 5 finding that updated (or
  reinforced) it
- A statement of what additional evidence would change your mind

**YOUR RESPONSE:**





---
## Wrap-up: Save homework artifacts
---

In [ ]:
# GUIDED: Save homework results
homework_results = {
    "tier_deltas_int8_int4": tier_deltas,
    "split_val_ece": split_rows,
    "named_class_trajectories": trajectory_df.to_dict(orient="records"),
    "constraint_check": check_df.to_dict(orient="records"),
}
with open("week5_homework_results.json", "w") as f:
    json.dump(homework_results, f, indent=2)
print("Saved week5_homework_results.json")

---
## Optional appendix: try the in-class wow demo yourself

**Not graded. Skip if you're done.** This is here for the curious.

In lecture I told you the story of trying to load a 14B model at int4
on a single T4 and almost not being able to download it. If you want
to feel that for yourself — load Qwen2.5-14B-Instruct via Unsloth's
pre-quantized variant and run one inference — here's the recipe.

**Requirements**
- Kaggle: GPU T4 ×2 accelerator (we pin to one)
- HF_TOKEN secret set
- ~12 GB free in `/kaggle/working`
- About 3 minutes of runtime

Copy the code below into a fresh cell on a NEW Kaggle notebook (don't
add it to this homework — it'll mess up your VRAM accounting if you
already loaded encoder + decoder logits into this kernel).

```python
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HOME"] = "/kaggle/working/.hf_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.makedirs("/kaggle/working/.hf_cache", exist_ok=True)

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map={"": 0}).eval()

prompt = ("I was charged a $35 late fee even though I paid on time. "
          "Classify this complaint into one short category.")
chat = tok.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False, add_generation_prompt=True,
)
inputs = tok(chat, return_tensors="pt").to("cuda:0")
with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=80, do_sample=False,
        pad_token_id=tok.eos_token_id,
    )
print(tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))

vram_gb = torch.cuda.max_memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Peak VRAM: {vram_gb:.2f} / {total_gb:.2f} GB")
```

Things worth noticing when you run it:
- The download step pauses for a couple of minutes (the int4-on-disk
  variant is ~10 GB). That's the workaround for the 20 GB working-dir
  wall.
- Peak VRAM should land around 10 GB on a 16 GB T4 — the 14B fits.
- Token throughput will be slow (single-digit tokens/sec). That's the
  bnb-on-T4-for-inference effect we taught you to expect.

---
### Submit

1. Run the notebook end-to-end with your answers filled in.
2. Export to HTML: **File → Download as → HTML (.html)** in Jupyter,
   or on Kaggle, click the "..." menu → "Save Version" to keep a static
   copy and download the HTML.
3. Upload the HTML to Moodle.

**Due:** Wednesday morning before Week 6 class.